# 06 - Diagnosis

SARIMA(1,1,2)(1,1,1,52) is the best model built so far (6.61% MAPE on the 52-week
statewide holdout), narrowly ahead of Prophet (6.72%) and clearly ahead of the ML model
(8.44%) and Seasonal Naive (8.29%). A single aggregate MAPE hides where a model actually
struggles -- this notebook diagnoses that directly, the same way notebook 01 investigated
the October spike before trusting it, rather than accepting one summary number as the
whole story.

Plan:
1. Rebuild the final SARIMA model and its test-period forecast/residuals
2. Find the worst individual weeks by absolute % error, and check whether they cluster
   around a specific calendar pattern or are scattered
3. Check whether error grows with distance into the 52-week horizon (expected, since
   uncertainty compounds forward) versus being driven by specific calendar events not
   explained by horizon distance alone
4. Use the ML model's category-level results (notebook 05) to identify which categories
   are hardest to forecast -- SARIMA/Prophet never touched this, so it's the only
   available source of category-level diagnosis

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from statsmodels.tsa.statespace.sarimax import SARIMAX

df = pd.read_csv("../data/raw/weekly_sales_by_category.csv", parse_dates=["week_start"])
statewide = df.groupby("week_start")["total_sales_dollars"].sum().reset_index()
statewide.columns = ["week_start", "total_sales"]
statewide = statewide.sort_values("week_start").reset_index(drop=True)

TEST_WEEKS = 52
train = statewide.iloc[:-TEST_WEEKS].copy()
test = statewide.iloc[-TEST_WEEKS:].copy()
y_train = train["total_sales"].values
y_test = test["total_sales"].values

sarima_model = SARIMAX(y_train, order=(1, 1, 2), seasonal_order=(1, 1, 1, 52),
                        enforce_stationarity=True, enforce_invertibility=True)
sarima_fit = sarima_model.fit(disp=False)
sarima_forecast = sarima_fit.get_forecast(steps=52).predicted_mean

test = test.copy()
test["forecast"] = sarima_forecast
test["abs_pct_error"] = np.abs((test["total_sales"] - test["forecast"]) / test["total_sales"]) * 100
test["signed_error"] = test["total_sales"] - test["forecast"]
test["month"] = test["week_start"].dt.month
test["horizon_week"] = range(1, 53)

print("Reproduced MAPE:", round(test["abs_pct_error"].mean(), 2))

Reproduced MAPE: 6.61


In [3]:
print("=== Worst 10 weeks by absolute % error ===")
print(test.sort_values("abs_pct_error", ascending=False)[
    ["week_start", "total_sales", "forecast", "abs_pct_error", "signed_error"]
].head(10).to_string(index=False))

print()
print("=== MAPE by month ===")
print(test.groupby("month")["abs_pct_error"].mean().round(2))

print()
print("=== Mean signed error by month (negative = actual came in below forecast) ===")
print(test.groupby("month")["signed_error"].mean().round(0))

=== Worst 10 weeks by absolute % error ===
week_start  total_sales     forecast  abs_pct_error  signed_error
2025-06-30   6274953.98 8.379638e+06      33.541031 -2.104684e+06
2026-04-27   6406210.04 8.170437e+06      27.539317 -1.764226e+06
2025-07-07   8653760.03 7.298678e+06      15.658882  1.355082e+06
2025-06-23  10318000.64 8.794555e+06      14.764928  1.523445e+06
2026-01-12   6170989.25 7.003690e+06      13.493789 -8.327003e+05
2025-12-29   7748866.46 6.776882e+06      12.543567  9.719842e+05
2026-01-05   7385348.58 6.506940e+06      11.893930  8.784082e+05
2025-05-26   7631471.07 8.519344e+06      11.634363 -8.878730e+05
2026-01-26   6293512.63 7.017037e+06      11.496357 -7.235247e+05
2025-12-22   9019300.27 8.071393e+06      10.509764  9.479072e+05

=== MAPE by month ===
month
1      9.39
2      4.34
3      5.32
4      9.89
5      6.99
6     11.23
7      7.62
8      4.84
9      5.49
10     2.95
11     4.51
12     6.29
Name: abs_pct_error, dtype: float64

=== Mean signed error

## Two specific questions, not just one summary number

1. Does forecast accuracy actually get worse deeper into the 52-week horizon, the way
   the widening confidence band in notebook 03's plot suggested -- or is that band
   growth just a mechanical property of the model that doesn't track real difficulty?
2. Is the violent late-May/June/early-July swing (June 23 over-forecast by 14.8%, June
   30 under-forecast by 33.5%, July 7 over-forecast by 15.7%, all within three weeks)
   a one-off, or does this exact calendar window behave this way every year -- the same
   "investigate before treating as anomaly" standard from notebook 01's October check?

In [4]:
ci = sarima_fit.get_forecast(steps=52).conf_int(alpha=0.05)
test["ci_width"] = ci[:, 1] - ci[:, 0]

print("Correlation(horizon_week, abs_pct_error):", round(np.corrcoef(test["horizon_week"], test["abs_pct_error"])[0, 1], 3))
print("Correlation(horizon_week, ci_width):     ", round(np.corrcoef(test["horizon_week"], test["ci_width"])[0, 1], 3))
print()
print("CI width, week 1 vs week 52:", round(test['ci_width'].iloc[0]), "vs", round(test['ci_width'].iloc[-1]))

Correlation(horizon_week, abs_pct_error): -0.028
Correlation(horizon_week, ci_width):      0.992

CI width, week 1 vs week 52: 3391519 vs 9872094


In [5]:
statewide["wk_pct_change"] = statewide["total_sales"].pct_change() * 100

for yr in range(2017, 2026):
    sub = statewide[(statewide["week_start"] >= f"{yr}-05-10") & (statewide["week_start"] <= f"{yr}-07-20")]
    print(f"--- {yr} ---")
    print(sub[["week_start", "total_sales", "wk_pct_change"]].to_string(index=False))
    print()

--- 2017 ---
week_start  total_sales  wk_pct_change
2017-05-15   6188694.70      -2.631015
2017-05-22   7170148.66      15.858820
2017-05-29   4548748.83     -36.559909
2017-06-05   6796457.90      49.413787
2017-06-12   6599117.89      -2.903571
2017-06-19   5774869.34     -12.490284
2017-06-26   7351412.79      27.300071
2017-07-03   5701185.62     -22.447756
2017-07-10   6180513.85       8.407518
2017-07-17   5964255.22      -3.499040

--- 2018 ---
week_start  total_sales  wk_pct_change
2018-05-14   6179263.90      -5.867213
2018-05-21   7324087.25      18.526856
2018-05-28   5222821.12     -28.689802
2018-06-04   6919399.27      32.483941
2018-06-11   6773170.02      -2.113323
2018-06-18   5681719.86     -16.114318
2018-06-25   8455901.90      48.826449
2018-07-02   4156343.28     -50.846837
2018-07-09   6629508.95      59.503402
2018-07-16   6109760.87      -7.839918

--- 2019 ---
week_start  total_sales  wk_pct_change
2019-05-13   7586454.23       8.116600
2019-05-20   7281422.19

In [6]:
cat_totals = df[df["category_group"] != "ALL_OTHER"].groupby("category_group")["total_sales_dollars"].sum().sort_values(ascending=False)
print("Total revenue by category:")
print(cat_totals)
print()

weekly_by_cat = df[df["category_group"] != "ALL_OTHER"].groupby(["category_group", "week_start"])["total_sales_dollars"].sum().reset_index()
cv = weekly_by_cat.groupby("category_group")["total_sales_dollars"].agg(["mean", "std"])
cv["cv_pct"] = (cv["std"] / cv["mean"] * 100).round(2)
print("Coefficient of variation (week-to-week volatility relative to each category's own scale):")
print(cv.sort_values("cv_pct"))

Total revenue by category:
category_group
AMERICAN VODKAS              5.463343e+08
CANADIAN WHISKIES            4.223723e+08
STRAIGHT BOURBON WHISKIES    2.964042e+08
WHISKEY LIQUEUR              2.138025e+08
100% AGAVE TEQUILA           2.132307e+08
Name: total_sales_dollars, dtype: float64

Coefficient of variation (week-to-week volatility relative to each category's own scale):
                                   mean            std  cv_pct
category_group                                                
WHISKEY LIQUEUR            4.390195e+05   99143.636657   22.58
AMERICAN VODKAS            1.121836e+06  256358.918038   22.85
CANADIAN WHISKIES          8.672942e+05  209845.762466   24.20
STRAIGHT BOURBON WHISKIES  6.086328e+05  199609.843055   32.80
100% AGAVE TEQUILA         4.378454e+05  199582.634074   45.58


## Summary

**Where SARIMA actually struggles (52-week holdout, 2025-05-05 to 2026-04-27):**

| Finding | Detail |
|---|---|
| Worst single week | 2025-06-30, 33.5% error (actual $6.27M vs. forecast $8.38M) |
| Worst month | June, 11.23% average MAPE (vs. 2.95% in October -- the pattern notebook 01 specifically investigated) |
| Horizon vs. accuracy | No correlation (r=-0.028) -- the widening confidence band does not predict where errors occur |
| Recurring failure mode | A violent Memorial Day-to-July 4th swing, present in all 9 years of history, that a fixed 52-week seasonal lag can't phase-align to |

**Category-level (from notebook 05's ML model, the only model with category forecasts):**

| Category | MAPE | Coefficient of variation |
|---|---|---|
| Whiskey Liqueur | 8.94% | 22.58% |
| Straight Bourbon Whiskies | 10.82% | 32.80% |
| American Vodkas | 13.57% | 22.85% |
| Canadian Whiskies | 13.69% | 24.20% |
| 100% Agave Tequila | 16.96% | 45.58% |

Takeaways:

1. **The single worst error and the single best month aren't where the earlier plot suggested.** Notebook 03's chart created an impression that the Dec 2025-Jan 2026 trough was the main weak spot; the actual worst miss (June 30, 33.5%) and worst month (June, 11.23%) are elsewhere entirely, and October -- the pattern this project specifically investigated in notebook 01 (decision #5) -- is where the model performs best (2.95% MAPE). Investigating anomalies pays off in the model, not just the raw data.
2. **A model's own uncertainty estimate isn't the same as knowing where it will actually be wrong.** The confidence band widens almost perfectly with horizon (r=0.992), but real error doesn't (r=-0.028) -- a reminder to check this kind of thing directly rather than reading it off a chart.
3. **The recurring Memorial Day/July 4th volatility is a structural limitation, not a tuning failure.** No `(p,d,q)(P,D,Q,s)` search fixes a model whose seasonal assumption (fixed week-of-year) is the wrong shape for a driver whose calendar position moves year to year. This is a concrete, well-understood lever for future work (explicit holiday-date modeling), not a vague "needs more tuning."
4. **Category difficulty is explained by intrinsic volatility only at the extremes**, and the middle three categories are a genuine open question rather than a clean pattern -- reported honestly rather than forced into a tidier story than the data supports.